## Gold Work Incremental

### Step 1 - Import and Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, UTC
import uuid

In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists gold_schema")

gold_run_id = str(uuid.uuid4())
print("Current Gold run ID = ", gold_run_id)

run_ts_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("Run Timestamp Folder = ", run_ts_str)

run_date_str = datetime.now().strftime("%Y-%m-%d")
print("run_date_str = ", run_date_str)

### Step 2 - Gold Control Table

In [0]:
spark.sql("""
          create table if not exists novacart_adb.gold_schema.process_control
          (
              layer string,
              entity_name string,
              last_processed_silver_run_id string,
              last_processed_silver_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          )
          using delta
          """)

### Step 3 - Helper Functions
This defines the reusable gold functions
- upsert_to_gold() - merges data into gold current state tables
- get_last_processed_silver_ts() - reads the gold watermark from the control table
- upsert_gold_control() - updates gold control after a successful run

In [0]:
def upsert_to_gold(source_df, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("trg").merge(source_df.alias("src"), f"trg.{join_key} = src.{join_key}")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
    else:
        source_df.write.format('delta').saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl = spark.table("novacart_adb.gold_schema.process_control")\
            .filter(col('layer') == 'gold')\
            .filter(col('entity_name') == entity_name)\
            .filter(col('run_status') == 'Success')\
            .orderBy(col('updated_at').desc())\
            .limit(1)
    
    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]['last_processed_silver_ingested_at']

In [0]:
def upsert_gold_ctrl(entity_name, last_processed_silver_run_id, last_processed_silver_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "gold",
                entity_name,
                last_processed_silver_run_id,
                last_processed_silver_ingested_at,
                int(rows_merged),
                "Success",
                gold_run_id,
                datetime.now()
            )
        ],
                schema = """
                layer string,
                entity_name string,
                last_processed_silver_run_id string,
                last_processed_silver_ingested_at timestamp,
                rows_merged bigint,
                run_status string,
                gold_run_id string,
                updated_at timestamp  
                """
    )

    dt = DeltaTable.forName(spark, "novacart_adb.gold_schema.process_control")
    (dt.alias("trg").merge(ctrl_df.alias("src"), "trg.layer = src.layer and trg.entity_name = src.entity_name")\
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "src.last_processed_silver_run_id",
            "last_processed_silver_ingested_at": "src.last_processed_silver_ingested_at",
            "rows_merged": "src.rows_merged",
            "run_status": "src.run_status",
            "gold_run_id": "src.gold_run_id",
            "updated_at": "src.updated_at"
        })\
        .whenNotMatchedInsertAll()\
        .execute())
    

### Step 4 - Read changed Silver rows only
For Gold Incremental processing

In [0]:
last_gold_ts = get_last_processed_silver_ts('orders_information')

print("Last processed Silver timestamp for Gold - ", last_gold_ts)

silver_orders_current = spark.read.table("novacart_adb.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_adb.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_adb.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current

else:
    changed_orders = silver_orders_current.filter(col('updated_at') > lit(last_gold_ts))
    changed_products = silver_products_current.filter(col('updated_at') > lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(col('processed_at') > lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("Number of changed orders = ", changed_orders_count)
print("Number of changed products = ", changed_products_count)
print("Number of changed payments = ", changed_payments_count)

### Step 5 - Find Impacted Order Ids
Gold is built on order grain, so if anything changes in orders, products or payments. We identify which order_id value has impacted.

Only those order IDs are rebuilt in Gold.

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()

# order_id is the foreign key in payments table
impacted_from_payments = changed_payments.select("order_id").distinct()

# product_id is foreign key in orders table which is primary key in products table
impacted_from_products = changed_products.alias('p').join(silver_orders_current.alias('o'), col('p.product_id') == col('o.product_id'), "inner").select(col("o.order_id")).distinct()

impacted_order_ids = (
    impacted_from_orders.union(impacted_from_payments).union(impacted_from_products).distinct()
)

print("Impacted order ids - ", impacted_order_ids.count())
display(impacted_order_ids.orderBy(col("order_id")))

### Step 6 - Build Gold Delta for impacted orders
This cell joins the impacted orders with the current silver products and payments tables, derives business columns and build the Gold Delta that will be merged into the gold current state table

In [0]:
impacted_orders = silver_orders_current.alias('o').join(impacted_order_ids.alias('i'), "order_id", "inner")

gold_delta = (
    impacted_orders.alias("o").join(silver_products_current.alias("p"), col("o.product_id") == col("p.product_id"), "inner")\
                                .join(silver_payments_current.alias("py"), col("o.order_id") == col("py.order_id"), "inner")\
                                .select(
                                    col("o.order_id"),
                                    col("o.customer_id"),
                                    col("p.product_id"),
                                    col("p.product_name"),
                                    col("p.category"),
                                    col("p.price").alias("product_price"),
                                    col("o.order_status"),
                                    col("o.order_amount"),
                                    col("py.payment_id"),
                                    col("py.payment_status"),
                                    col("py.paid_amount"),
                                    col("o.order_date"),
                                    col("o.order_month"),
                                    col("o.order_year"),
                                    greatest(col("o.updated_at").cast("timestamp"),
                                             col("p.updated_at").cast("timestamp"),
                                             col("py.processed_at").cast("timestamp")).alias("gold_update_ts")
                                ).dropDuplicates(["order_id"])\
                                .withColumn("payment_completion_ratio", 
                                            when(col("order_amount") > 0, col("paid_amount") / col("order_amount")).otherwise(lit(0.0)))\
                                .withColumn("payment_state", when(col("order_amount") == 0, "Invalid_order_amount")\
                                                            .when(col("payment_completion_ratio") == 1, "Paid")\
                                                            .when(col("payment_completion_ratio") == 0, "Unpaid")\
                                                            .when(col("payment_completion_ratio") < 1, "Partially_paid")\
                                                            .when(col("payment_completion_ratio") > 1, "Over_paid"))\
                                .withColumn("gold_updated_date", to_date(col("gold_update_ts")))\
                                .withColumn("gold_run_id", lit(gold_run_id))
)

print("gold_delta_rows - ", gold_delta.count())
display(gold_delta)

### Step 7 - Merge gold current state table
If Gold Delta contains rows, it'll merges them into gold_schema.orders_information

If there is no impacted rows, nothing is merged

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novacart_adb.gold_schema.orders_information", "order_id")
else:
    print("No new records to insert to gold table")

In [0]:
%sql
select * from novacart_adb.gold_schema.orders_information

### Step 8 - Maintain Gold SCD Type 2 history

In [0]:
if not spark.catalog.tableExists("novacart_adb.gold_schema.orders_information_scd_2"):
    spark.sql("""
              create table novacart_adb.gold_schema.orders_information_scd_2
              using delta
              as
              select *,
                     cast(null as timestamp) valid_from_ts,
                     cast(null as timestamp) valid_to_ts,
                     true as is_current
                     from novacart_adb.gold_schema.orders_information
                     where 1 == 0
              """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into novacart_adb.gold_schema.orders_information_scd_2 as t
              using gold_delta_view as s
              on t.order_id = s.order_id and t.is_current = true
              when matched and (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
                )

            then update set 
                is_current = false,
                valid_to_ts = s.gold_update_ts
            """)
    
    spark.sql("""
              insert into novacart_adb.gold_schema.orders_information_scd_2
              select s.*,
                     s.gold_update_ts as valid_from_ts,
                     cast(null as timestamp) valid_to_ts,
                     true as is_current
                from gold_delta_view s
                left join novacart_adb.gold_schema.orders_information_scd_2 t
                on s.order_id = t.order_id and t.is_current = true
                where t.order_id is null or (
                    not(t.order_status <=> s.order_status) or
                    not(t.order_amount <=> s.order_amount) or
                    not(t.paid_amount <=> s.paid_amount) or
                    not(t.payment_id <=> s.payment_id) or
                    not(t.category <=> s.category) or
                    not(t.product_name <=> s.product_name) or
                    not(t.product_price <=> s.product_price)
                )
            
              """)

### Step 9 - update category level Gold Aggregation

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta.select("category").filter(col("category").isNotNull()).distinct()
    )

    category_performance_delta = (
        spark.read.table("novacart_adb.gold_schema.orders_information")\
            .join(impacted_categories, "category", "inner")\
            .groupBy("category")
            .agg(
                countDistinct("order_id").alias("total_orders"),
                sum(
                    when(col("order_amount") > 0, col("order_amount")).otherwise(lit(0.0))
                ).alias("gross_merchandise_value"),
                sum(
                    when(col("paid_amount") > 0, col("paid_amount")).otherwise(lit(0.0))
                ).alias("total_paid_amount"),
                avg(col("payment_completion_ratio")).alias("payment_completion_ratio"),
                (
                    sum(when(col("payment_status") == "FAILED", 1).otherwise(0)) / countDistinct("order_id")
                ).alias("payment_failure_rate")
                
            )             
    )
    upsert_to_gold(category_performance_delta, "novacart_adb.gold_schema.category_performance", "category")

In [0]:
%sql
select * from novacart_adb.gold_schema.category_performance

### Step 10 - Publish Gold snapshots to Volume
This cell writes two kinds of Gold outputs to a Databrick volume
- latest snapshot - overwrite every successful run
- timestamped historical snapshot - a new folder for each successful run

This is useful for audit, rollback and teaching demos.

In [0]:
spark.sql("create volume if not exists novacart_adb.gold_schema.gold_snapshot_vol")

In [0]:
latest_orders_path = (
    "/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_latest/category_performance"
)

historical_orders_path = f"/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"

historical_category_path = f"/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

spark.read.table("novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)

spark.read.table("novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)

spark.read.table("novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print(f"latest_orders_path: {latest_orders_path}")
print(f"latest_category_path: {latest_category_path}")
print(f"historical_orders_path: {historical_orders_path}")
print(f"historical_category_path: {historical_category_path}")

### Step 11 - Update Gold control table

In [0]:
latest_silver_ts = silver_orders_current.agg(max("bronze_ingested_at").alias("mx")).collect()[0]['mx']

latest_silver_run_id = (silver_orders_current.filter(col("bronze_ingested_at") == latest_silver_ts).agg(max("silver_run_id").alias("mx")).collect()[0]['mx']) if latest_silver_ts is not None else None

upsert_gold_ctrl("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())
display(spark.table("novacart_adb.gold_schema.process_control"))

In [0]:
%sql
select * from novacart_adb.gold_schema.process_control